In [ ]:
pip install accelerate

In [2]:
import pandas as pd
import torch
import random
from tqdm.auto import tqdm
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import ast


In [ ]:
MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto"
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

def llm_generate(prompt, max_new_tokens=128):
    out = generator(prompt, max_new_tokens=max_new_tokens, do_sample=False)
    text = out[0]["generated_text"]
    return text[len(prompt):].strip() if text.startswith(prompt) else text.strip()


In [ ]:
test_df=pd.read_csv("filtered_test.csv")
test_df

In [15]:
def safe_parse(x):
    if isinstance(x, str):
        return ast.literal_eval(x)
    return x

test_df["text_nodes"] = test_df["text_nodes"].apply(safe_parse)
test_df["interactions"] = test_df["interactions"].apply(safe_parse)

docs = pd.read_csv("datasets_documents.csv")
# resp = pd.read_csv("response.csv")
# dial = pd.read_csv("dialogue.csv")
summ = pd.read_csv("summaries.csv")

doc_map  = dict(zip(docs["NewsID"], docs["NewsBody"]))
# resp_map = dict(zip(resp["response_id"], resp["response_text"]))
# dial_map = dict(zip(dial["dialogue_id"], dial["dialogue_text"]))
summ_map = {
    (row["UserID"], row["NewsID"]): row["Summary"]
    for _, row in summ.iterrows()
}



In [16]:
def resolve_node(node_id):
    if node_id.startswith("d-"):  return doc_map.get(node_id, "")
    # if node_id.startswith("r-"):  return resp_map.get(node_id, "")
    # if node_id.startswith("dl-"): return dial_map.get(node_id, "")
    if node_id.startswith("s-"):  return summ_map.get(node_id, "")
    return ""

def build_history(row):
    history = []
    for node, act in zip(row["text_nodes"], row["interactions"]):
        text = resolve_node(node)
        history.append(text)
    return "\n".join(history)


In [ ]:
up_df = test_df[test_df["task"]=="personalized_summarization"].copy()

user_histories = {}

for uid, g in tqdm(up_df.groupby("userid")):
    texts = []
    for _, row in g.iterrows():
        texts.append(build_history(row))
    user_histories[uid] = "\n".join(texts[:300])


In [30]:
def make_personalized_summary_prompt(history, original_summary):
    return f"""
You are given a user's preference history and an original news summary.

Rewrite the summary to better match the user's interests.
The output must be a single personalized summary of 15–20 words.

User preference history:
{history}

Original news summary:
{original_summary}

Write ONLY the personalized summary below.
Do not explain. Do not use placeholders. Do not repeat instructions.
""".strip()


In [31]:
def extract_summary(prompt, generated_text):
    text = generated_text
    summary = text[len(prompt):].strip() if text.startswith(prompt) else text.strip()
    return summary.split("\n")[0].strip()


In [ ]:
print(up_df.columns.tolist())


In [33]:
summ_df = pd.read_csv("summaries.csv")


In [34]:
def truncate_text(text, max_chars=4000):
    if not isinstance(text, str):
        return ""
    return text[:max_chars]


In [ ]:
rows = []

for _, row in tqdm(up_df[:5].iterrows(), total=len(up_df[:5])):
    uid = row["userid"]
    qid = row["query_text_id"]

    original_summary = summ_map.get((uid, qid))
    if original_summary is None:
        continue

    history = truncate_text(build_history(row), max_chars=4000)

    prompt = make_personalized_summary_prompt(history, original_summary)
    raw = llm_generate(prompt, max_new_tokens=32)
    summary = extract_summary(prompt, raw)

    rows.append({
        "userid": uid,
        "queryid": qid,
        "summary": summary,
        "original_summary": original_summary
    })


pd.DataFrame(rows).to_csv(
    "personalized_summarization_outputs.csv",
    index=False
)


In [ ]:
pred_df=pd.read_csv("/teamspace/studios/this_studio/personalized_summarization_outputs.csv")
pred_df